# ReServe memory supplement — FP16 and NF4

Use this **separate Kaggle notebook**, once per mode. Keep your completed timing notebooks and results unchanged.

It measures model-resident allocation, actual KV tensors, one-token cache growth, operation peaks, and memory after request cleanup. It **does not repeat your 50-trial timing sweeps** or implement live precision switching.

**Kaggle:** enable Internet and a Tesla T4 GPU. A T4 ×2 allocation is fine; this notebook uses GPU 0 only. Do not run another model in the same kernel.

The new measurements use the exact checkpoint recorded by your NF4 run, `989aa7980e4cf806f80c7fef2b1adb7bc71aa306`. Your original FP16 timing revision remains unrecorded; this supplement does not change that history.

## 1. Install dependencies

Run the next cell once. Then **restart the Python kernel/session** so Transformers sees the installed packages. After the restart, continue at Step 2; the helper code below is self-contained. Do not upgrade PyTorch or CUDA.


In [ ]:
%pip install -q "transformers==5.20.2" "bitsandbytes==0.50.2" "accelerate==1.15.0"


## 2. Choose a mode

First run: leave `MODE = "fp16"`. Run Steps 2–5 in order and download the ZIP.

Second run: **restart the Python kernel**, change only `MODE` to `"nf4"`, then run Steps 2–5 again. A fresh kernel prevents the two models or old tensors from affecting the baseline.

The default 16 shapes match your timing grid. Each gets three fresh request lifecycles: prompt → one decode step → full cleanup. There are 48 diagnostic trials per mode, not 1,600 timing trials. Every new run gets a different output directory.


In [ ]:
MODE = "fp16"                 # First "fp16"; after downloading/restarting, use "nf4"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
BATCH_SIZES = [1, 2, 4, 8]
LENGTHS = [256, 512, 1024, 2048]
MEMORY_REPEATS = 3


## 3. Define the measurement code

Run this whole cell. It only defines helpers; Step 4 loads the selected model and starts the pass.

The code checks your T4, PyTorch, CUDA runtime, Transformers, actual KV dtype/shape, and (for NF4) the 196 converted linear layers. An environment mismatch stops with an explanation instead of silently changing your setup.


In [ ]:
"""Memory-only diagnostics for ReServe. Embedded in the delivered notebook."""
import gc
import itertools
import json
import os
import platform
import random
import statistics
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, BitsAndBytesConfig


def host_memory():
    """CPU RSS and available system RAM, in bytes; optional Linux observations."""
    result = {"host_rss_bytes": None, "host_available_bytes": None}
    for filename, field, target in [
        ("/proc/self/status", "VmRSS:", "host_rss_bytes"),
        ("/proc/meminfo", "MemAvailable:", "host_available_bytes"),
    ]:
        try:
            for line in Path(filename).read_text().splitlines():
                if line.startswith(field):
                    result[target] = int(line.split()[1]) * 1024
                    break
        except (OSError, ValueError):
            pass
    return result


def observe(device):
    """Synchronized scalars only: never return a live GPU tensor."""
    torch.cuda.synchronize(device)
    free, total = torch.cuda.mem_get_info(device)
    allocated = int(torch.cuda.memory_allocated(device))
    reserved = int(torch.cuda.memory_reserved(device))
    return {
        "allocated_bytes": allocated,
        "reserved_bytes": reserved,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated(device)),
        "peak_reserved_bytes": int(torch.cuda.max_memory_reserved(device)),
        "device_free_bytes": int(free),
        "device_total_bytes": int(total),
        "device_global_used_bytes": int(total - free),
        # Diagnostic only: includes the CUDA context and other allocations/users.
        "device_used_minus_torch_reserved_bytes": int(total - free - reserved),
        **host_memory(),
    }


def inspect_cache(cache, batch_size, length, config):
    """Verify actual KV tensors; deduplicate backing storage by device/pointer."""
    if cache is None or not hasattr(cache, "layers"):
        raise ValueError("Expected a Transformers cache exposing .layers")
    layers = list(cache.layers)
    if len(layers) != config.num_hidden_layers:
        raise ValueError(f"Expected {config.num_hidden_layers} cache layers, got {len(layers)}")
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    expected_shape = (batch_size, config.num_key_value_heads, length, head_dim)
    logical_bytes, unique_storages, details = 0, {}, []
    for layer_index, layer in enumerate(layers):
        for kind in ("keys", "values"):
            tensor = getattr(layer, kind)
            if tuple(tensor.shape) != expected_shape:
                raise ValueError(f"Layer {layer_index} {kind}: {tuple(tensor.shape)} != {expected_shape}")
            if tensor.dtype != torch.float16 or str(tensor.device) != "cuda:0":
                raise ValueError(f"Expected FP16 KV on cuda:0; got {tensor.dtype}, {tensor.device}")
            logical = int(tensor.numel() * tensor.element_size())
            storage = tensor.untyped_storage()
            storage_key = (str(tensor.device), int(storage.data_ptr()))
            backing = int(storage.nbytes())
            shared = storage_key in unique_storages
            unique_storages[storage_key] = backing
            logical_bytes += logical
            details.append({
                "layer": layer_index, "kind": kind,
                "shape": json.dumps(list(tensor.shape)), "dtype": str(tensor.dtype),
                "logical_bytes": logical, "backing_storage_bytes": backing,
                "storage_seen_previously": shared,
            })
    expected_bytes = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2 * batch_size * length
    if logical_bytes != expected_bytes:
        raise ValueError(f"KV size {logical_bytes} != architecture-derived {expected_bytes}")
    return {
        "kv_logical_bytes": logical_bytes,
        "kv_unique_backing_bytes": sum(unique_storages.values()),
        "kv_unique_storages": len(unique_storages),
        "kv_bytes_per_cached_token": logical_bytes // (batch_size * length),
        "kv_dtype": "float16", "cached_tokens_per_request": length,
    }, details


def check_growth(before, after, batch_size, bytes_per_token):
    growth = after["kv_logical_bytes"] - before["kv_logical_bytes"]
    expected = batch_size * bytes_per_token
    if growth != expected:
        raise ValueError(f"One decode step grew KV by {growth} bytes; expected {expected}")
    return growth


def non_kv_upper_bound(peak, baseline, minimum_live_kv):
    """Loose bound from this trial, NOT an isolated workspace measurement."""
    return max(0, int(peak) - int(baseline) - int(minimum_live_kv))


def write_checkpoint(directory, metadata, snapshots, trials, cache_details):
    """Atomic file replacements keep earlier measurements on failure/interruption."""
    frames = {
        "memory_snapshots.csv": snapshots,
        "memory_trials.csv": trials,
        "cache_tensors.csv": cache_details,
    }
    for name, rows in frames.items():
        if not rows:
            continue
        temporary = directory / (name + ".tmp")
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(directory / name)
    temporary = directory / "metadata.json.tmp"
    temporary.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    temporary.replace(directory / "metadata.json")


@torch.inference_mode()
def warmup_model(model, device, batch_size, length, token_id):
    """Initialize representative kernels; leave no request/cache references."""
    ids = mask = prefix = cache = token = decode_mask = decoded = None
    try:
        ids = torch.full((batch_size, length), token_id, dtype=torch.long, device=device)
        mask = torch.ones_like(ids)
        prefix = model(input_ids=ids, attention_mask=mask, use_cache=True, logits_to_keep=1)
        cache = prefix.past_key_values
        inspect_cache(cache, batch_size, length, model.config)
        token = prefix.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        decode_mask = torch.ones((batch_size, length + 1), dtype=torch.long, device=device)
        decoded = model(input_ids=token, attention_mask=decode_mask,
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
        inspect_cache(decoded.past_key_values, batch_size, length + 1, model.config)
        if not torch.isfinite(prefix.logits).all().item() or not torch.isfinite(decoded.logits).all().item():
            raise ValueError("Warm-up produced non-finite logits")
        torch.cuda.synchronize(device)
    finally:
        ids = mask = prefix = cache = token = decode_mask = decoded = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device)


@torch.inference_mode()
def measure_request(model, device, batch_size, length, token_id, identity,
                    snapshots, cache_details, baseline, bytes_per_token, tolerance):
    """One fresh prompt, ONE cached decode step, then complete request release."""
    ids = prompt_mask = decode_mask = prefix = cache = next_token = decoded = None
    row = {**identity, "status": "started", "last_stage": "baseline"}

    def snap(stage, kv=None, peak_scope="none"):
        observation = observe(device)
        snapshots.append({**identity, "stage": stage, "peak_scope": peak_scope,
                          **observation, **(kv or {})})
        row["last_stage"] = stage
        return observation

    def record_cache(stage, current_cache, cached_length):
        info, details = inspect_cache(current_cache, batch_size, cached_length, model.config)
        cache_details.extend({**identity, "stage": stage, **detail} for detail in details)
        return info

    try:
        ids = torch.full((batch_size, length), token_id, dtype=torch.long, device=device)
        prompt_mask = torch.ones_like(ids)
        decode_mask = torch.ones((batch_size, length + 1), dtype=torch.long, device=device)
        inputs_ready = snap("inputs_ready")
        torch.cuda.reset_peak_memory_stats(device)
        prefix = model(input_ids=ids, attention_mask=prompt_mask, use_cache=True, logits_to_keep=1)
        prefill_peak = snap("prefill_returned", peak_scope="prefill")
        cache = prefix.past_key_values
        prefill_kv = record_cache("after_prefill", cache, length)

        next_token = prefix.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        prefix = None
        before_decode = snap("cache_retained_before_decode", prefill_kv)
        torch.cuda.reset_peak_memory_stats(device)
        decoded = model(input_ids=next_token, attention_mask=decode_mask,
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
        decode_peak = snap("decode_returned", peak_scope="decode")
        cache = decoded.past_key_values
        decode_kv = record_cache("after_decode", cache, length + 1)
        growth = check_growth(prefill_kv, decode_kv, batch_size, bytes_per_token)

        # Retain ONLY the cache, with model weights still resident.
        ids = prompt_mask = decode_mask = next_token = decoded = None
        kv_only = snap("only_cache_retained", decode_kv)
        row.update({
            "baseline_allocated_bytes": baseline["allocated_bytes"],
            "baseline_reserved_bytes": baseline["reserved_bytes"],
            "inputs_ready_allocated_bytes": inputs_ready["allocated_bytes"],
            "prefill_peak_allocated_bytes": prefill_peak["peak_allocated_bytes"],
            "prefill_peak_reserved_bytes": prefill_peak["peak_reserved_bytes"],
            "prefill_return_allocated_bytes": prefill_peak["allocated_bytes"],
            "prefill_kv_logical_bytes": prefill_kv["kv_logical_bytes"],
            "prefill_kv_unique_backing_bytes": prefill_kv["kv_unique_backing_bytes"],
            "before_decode_allocated_bytes": before_decode["allocated_bytes"],
            "decode_peak_allocated_bytes": decode_peak["peak_allocated_bytes"],
            "decode_peak_reserved_bytes": decode_peak["peak_reserved_bytes"],
            "decode_return_allocated_bytes": decode_peak["allocated_bytes"],
            "decode_kv_logical_bytes": decode_kv["kv_logical_bytes"],
            "decode_kv_unique_backing_bytes": decode_kv["kv_unique_backing_bytes"],
            "kv_bytes_per_cached_token": decode_kv["kv_bytes_per_cached_token"],
            "decode_growth_bytes": growth,
            "kv_only_allocated_bytes": kv_only["allocated_bytes"],
            # Prefill starts with zero KV; decode starts with the existing prompt KV.
            # These upper bounds include inputs/logits/allocator overhead. Prefill's
            # bound is particularly loose because KV itself grows during execution.
            "prefill_non_kv_upper_bound_bytes": non_kv_upper_bound(
                prefill_peak["peak_allocated_bytes"], baseline["allocated_bytes"], 0),
            "decode_non_kv_upper_bound_bytes": non_kv_upper_bound(
                decode_peak["peak_allocated_bytes"], baseline["allocated_bytes"],
                prefill_kv["kv_logical_bytes"]),
            "status": "ok",
        })
    except BaseException as exc:
        row["failure_stage"] = row["last_stage"]
        row["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else (
            "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) else "error")
        row["error"] = f"{type(exc).__name__}: {str(exc).splitlines()[0] if str(exc) else ''}"
        raise
    finally:
        # No tensor or exception object is stored in the result records.
        ids = prompt_mask = decode_mask = prefix = cache = next_token = decoded = None
        gc.collect()
        released = snap("request_released_pool_kept")
        torch.cuda.empty_cache()
        trimmed = snap("request_released_pool_trimmed")
        difference = released["allocated_bytes"] - baseline["allocated_bytes"]
        row.update({
            "released_allocated_bytes": released["allocated_bytes"],
            "released_reserved_bytes": released["reserved_bytes"],
            "trimmed_allocated_bytes": trimmed["allocated_bytes"],
            "trimmed_reserved_bytes": trimmed["reserved_bytes"],
            "released_minus_baseline_bytes": difference,
            "cleanup_within_tolerance": abs(difference) <= tolerance,
        })
        # Caller receives a scalar checkpoint even if an exception is re-raised.
        identity["_trial_result"] = row
    return row


def build_summaries(trials):
    """Keep one row per supported shape; never subtract unrelated maxima."""
    groups = {}
    for row in trials:
        groups.setdefault((row["batch_size"], row["length_tokens"]), []).append(row)
    result = []
    for (batch_size, length), group in sorted(groups.items()):
        ok = [r for r in group if r["status"] == "ok"]
        item = {"batch_size": batch_size, "length_tokens": length,
                "trials_recorded": len(group), "ok_trials": len(ok),
                "status": "ok" if len(ok) == len(group) else "incomplete_or_error"}
        if ok:
            for column in (
                "baseline_allocated_bytes", "prefill_peak_allocated_bytes",
                "decode_peak_allocated_bytes", "prefill_kv_logical_bytes",
                "decode_kv_logical_bytes", "prefill_non_kv_upper_bound_bytes",
                "decode_non_kv_upper_bound_bytes",
            ):
                item[column + "_max"] = max(r[column] for r in ok)
            item["baseline_allocated_bytes_median"] = statistics.median(r["baseline_allocated_bytes"] for r in ok)
            item["max_absolute_cleanup_delta_bytes"] = max(abs(r["released_minus_baseline_bytes"]) for r in ok)
            item["all_cleanup_checks_pass"] = all(r["cleanup_within_tolerance"] for r in ok)
        result.append(item)
    return result


def run_memory_supplement(mode, model_revision, batches, lengths, repeats=3):
    """One fresh-kernel mode per invocation; no latency sweep or live switches."""
    if mode not in ("fp16", "nf4"):
        raise ValueError('MODE must be "fp16" or "nf4"')
    if not batches or not lengths or repeats < 1:
        raise ValueError("Choose nonempty positive batch/length lists and repeats >= 1")
    if any(not isinstance(x, int) or isinstance(x, bool) or x <= 0 for x in batches + lengths):
        raise ValueError("Batch sizes and lengths must be positive integers")
    if len(set(batches)) != len(batches) or len(set(lengths)) != len(lengths):
        raise ValueError("Duplicate batch sizes or lengths")
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle T4 GPU first")
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
    expected = {"gpu": "Tesla T4", "torch": "2.11.0+cu130", "cuda": "13.0", "transformers": "5.20.2"}
    actual = {"gpu": torch.cuda.get_device_name(device), "torch": torch.__version__,
              "cuda": torch.version.cuda, "transformers": transformers.__version__}
    mismatches = {k: (actual[k], v) for k, v in expected.items() if actual[k] != v}
    if mismatches:
        raise RuntimeError(f"Environment differs from your timing profiles: {mismatches}. Do not upgrade PyTorch to bypass this check.")
    torch.cuda.synchronize(device)
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.memory_allocated(device) != 0:
        raise RuntimeError("GPU tensors already exist. Restart the Python kernel; rerun this notebook from the settings cell.")

    model_id, token_id = "Qwen/Qwen2.5-1.5B-Instruct", 14990
    run_id = f"memory_{mode}_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = Path("/kaggle/working/reserve_memory") / run_id
    directory.mkdir(parents=True, exist_ok=False)
    snapshots, trials, details = [], [], []
    before_load = observe(device)
    snapshots.append({"run_id": run_id, "mode": mode, "stage": "before_model_load", **before_load})
    metadata = {
        "schema_version": 1, "run_id": run_id, "mode": mode, "run_status": "loading",
        "model_id": model_id, "requested_revision": model_revision,
        "reference_nf4_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
        "reference_fp16_run_id": "fp16_20261126T204405_157396Z",
        "reference_nf4_run_id": "bnb_nf4_fp16_compute_kv_20261126T220629_182036Z",
        "original_fp16_revision": None,
        "revision_note": "This new supplement pins the recorded NF4 revision for both modes. The old FP16 timing revision remains unknown.",
        "pid": os.getpid(), "python": platform.python_version(), "environment": actual,
        "gpu_total_memory_bytes": torch.cuda.get_device_properties(device).total_memory,
        "packages": {name: version(name) for name in ("torch", "transformers", "bitsandbytes", "accelerate", "pandas")},
        "allocator_environment": {name: os.environ.get(name) for name in ("PYTORCH_ALLOC_CONF", "PYTORCH_CUDA_ALLOC_CONF", "CUDA_VISIBLE_DEVICES")},
        "attention_implementation_requested": "sdpa", "batch_sizes": batches,
        "lengths": lengths, "repeats": repeats, "input_token_id": token_id,
        "cleanup_tolerance_bytes": 1024**2,
        "expected_trial_count": len(batches) * len(lengths) * repeats,
        "units": "All memory columns are integer bytes; display converts to MiB.",
        "measurement_notes": [
            "Allocator allocated and reserved are overlapping views; never add them.",
            "Cache logical bytes and unique backing storage are also overlapping views.",
            "Device usage/free memory is a whole-device observation, not exclusively this process.",
            "Resident baseline includes persistent tensors/quantization state; this is not pure weight bytes.",
            "Cache inspection returns scalars; it does not intentionally keep cache tensors alive.",
            "Operation peaks are taken in the same trial as its baseline and cache checks.",
            "non_kv_upper_bound subtracts only a guaranteed minimum live KV from the operation peak-minus-baseline; it is not exact workspace.",
            "The prefill bound is deliberately loose: it includes cache construction. Do not add components to an already observed total peak.",
            "HF DynamicCache is tested; no paged allocator, shared-prefix pool, weight restoration or switch is implemented here.",
            "empty_cache observation is a diagnostic after request cleanup, not an assumed per-token production operation.",
            "Passes characterize memory for supported homogeneous shapes, not a final dynamic serving capacity.",
        ],
    }
    write_checkpoint(directory, metadata, snapshots, trials, details)
    print(f"Mode: {mode}; saving to {directory}", flush=True)
    print("This is a memory pass, not another 50-trial timing sweep.", flush=True)
    model = None
    try:
        options = dict(revision=model_revision, dtype=torch.float16, attn_implementation="sdpa")
        if mode == "nf4":
            import bitsandbytes as bnb
            if version("bitsandbytes") != "0.50.2" or version("accelerate") != "1.15.0":
                raise RuntimeError("Run the installation cell, restart the Python kernel, then rerun from settings")
            quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                              bnb_4bit_compute_dtype=torch.float16,
                                              bnb_4bit_use_double_quant=False)
            model = AutoModelForCausalLM.from_pretrained(
                model_id, quantization_config=quantization, device_map={"": 0}, **options).eval()
            count = sum(isinstance(module, bnb.nn.Linear4bit) for module in model.modules())
            if count != 196:
                raise ValueError(f"Expected 196 NF4 linear layers from your measured setup, got {count}")
            metadata.update(quantized_linear_layers=count, quantization_config=quantization.to_dict())
        else:
            model = AutoModelForCausalLM.from_pretrained(model_id, **options).to(device).eval()
            if getattr(model, "is_quantized", False):
                raise ValueError("FP16 mode unexpectedly loaded a quantized model")
        if any(p.device != device for p in model.parameters()):
            raise ValueError("All parameters must be on GPU 0")
        if any(p.dtype != torch.float16 for p in model.parameters() if p.is_floating_point()):
            raise ValueError("Remaining floating parameters must be FP16")
        if max(lengths) + 1 > model.config.max_position_embeddings:
            raise ValueError("Context exceeds the model's maximum")
        metadata["model_config"] = model.config.to_dict()
        head_dim = getattr(model.config, "head_dim", None) or model.config.hidden_size // model.config.num_attention_heads
        bytes_per_token = 2 * model.config.num_hidden_layers * model.config.num_key_value_heads * head_dim * 2
        if bytes_per_token != 28672:
            raise ValueError(f"Unexpected architecture-derived KV size: {bytes_per_token}")
        metadata["derived_kv_bytes_per_cached_token"] = bytes_per_token
        gc.collect()
        torch.cuda.empty_cache()
        after_load = observe(device)
        snapshots.append({"run_id": run_id, "mode": mode, "stage": "after_model_load", **after_load})
        metadata["after_model_load"] = after_load
        write_checkpoint(directory, metadata, snapshots, trials, details)

        warmup_shapes = sorted({(min(batches), min(lengths)), (max(batches), max(lengths))})
        for b, length in warmup_shapes:
            print(f"Warm-up: B={b}, L={length}", flush=True)
            warmup_model(model, device, b, length, token_id)
        baseline = observe(device)
        snapshots.append({"run_id": run_id, "mode": mode, "stage": "after_warmup_no_requests", **baseline})
        metadata["steady_baseline"] = baseline
        metadata["kv_dtype_verified"] = "float16"
        metadata["kv_bytes_per_cached_token_verified"] = bytes_per_token
        metadata["warmup_shapes"] = warmup_shapes
        metadata["run_status"] = "running"
        print(f"Steady allocated baseline: {baseline['allocated_bytes'] / 1024**2:.2f} MiB", flush=True)
        write_checkpoint(directory, metadata, snapshots, trials, details)

        cases = list(itertools.product(batches, lengths))
        random.Random(42).shuffle(cases)
        metadata["case_order"] = cases
        for batch_size, length in cases:
            for repeat in range(1, repeats + 1):
                gc.collect()
                torch.cuda.empty_cache()
                case_baseline = observe(device)
                identity = {"run_id": run_id, "mode": mode, "batch_size": batch_size,
                            "length_tokens": length, "repeat": repeat}
                snapshots.append({**identity, "stage": "before_request", **case_baseline})
                recoverable_oom = False
                try:
                    measure_request(model, device, batch_size, length, token_id, identity,
                                    snapshots, details, case_baseline, bytes_per_token,
                                    metadata["cleanup_tolerance_bytes"])
                except torch.cuda.OutOfMemoryError:
                    recoverable_oom = True
                finally:
                    record = identity.pop("_trial_result", None)
                    if record is not None:
                        trials.append(record)
                    metadata["trials_recorded"] = len(trials)
                    write_checkpoint(directory, metadata, snapshots, trials, details)
                if recoverable_oom:
                    print(f"OOM: B={batch_size} L={length}; saved diagnostics, moving to next shape", flush=True)
                    break
            current = [r for r in trials if r["batch_size"] == batch_size and r["length_tokens"] == length]
            good = [r for r in current if r["status"] == "ok"]
            if good:
                peak = max(r["prefill_peak_allocated_bytes"] for r in good) / 1024**2
                clean = all(r["cleanup_within_tolerance"] for r in good)
                print(f"B={batch_size:2} L={length:4}: {len(good)}/{repeats} trials; prefill peak={peak:.2f} MiB; cleanup={clean}", flush=True)

        summary = build_summaries(trials)
        pd.DataFrame(summary).to_csv(directory / "memory_summary.csv", index=False)
        ok = [r for r in trials if r["status"] == "ok"]
        full_coverage = len(ok) == metadata["expected_trial_count"] and len(trials) == len(ok)
        cleanup_ok = bool(ok) and all(r["cleanup_within_tolerance"] for r in ok)
        metadata["all_expected_trials_ok"] = full_coverage
        metadata["all_cleanup_checks_pass"] = cleanup_ok
        metadata["run_status"] = "completed" if full_coverage and cleanup_ok else "completed_needs_review"
        metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        write_checkpoint(directory, metadata, snapshots, trials, details)
        print(f"{metadata['run_status']}: {len(ok)} successful trials", flush=True)
        print("Download the ZIP from the next cell, then restart the kernel before changing modes.", flush=True)
        return pd.DataFrame(summary), directory
    except BaseException as exc:
        metadata["run_status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "error"
        metadata["error"] = f"{type(exc).__name__}: {str(exc)}"
        write_checkpoint(directory, metadata, snapshots, trials, details)
        print(f"Stopped; partial diagnostics are saved in {directory}", flush=True)
        raise
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()


## 4. Run the memory pass

Warm-up runs the smallest and largest selected shapes once. Diagnostics then record a baseline with no requests, create a fresh prompt cache, decode one token, retain only the cache, release it, and inspect memory before and after trimming unused allocator-pool memory.

Files are checkpointed after each trial. If a supported shape unexpectedly runs out of memory, its failed trial is retained and the next shape is attempted. Unexpected errors stop after saving diagnostics. A failed largest-shape warm-up stops the run: send its error and saved files for review.


In [ ]:
memory_summary_df, memory_output_dir = run_memory_supplement(
    mode=MODE,
    model_revision=MODEL_REVISION,
    batches=BATCH_SIZES,
    lengths=LENGTHS,
    repeats=MEMORY_REPEATS,
)

display_columns = [
    "batch_size", "length_tokens", "ok_trials", "status",
    "baseline_allocated_bytes_median",
    "prefill_peak_allocated_bytes_max", "decode_peak_allocated_bytes_max",
    "max_absolute_cleanup_delta_bytes", "all_cleanup_checks_pass",
]
readable = memory_summary_df[[c for c in display_columns if c in memory_summary_df]].copy()
for column in list(readable.columns):
    if "_bytes" in column:
        readable[column] = readable[column] / 1024**2
        readable = readable.rename(columns={column: column.replace("_bytes", "_MiB")})
display(readable.round(3))
print("Results:", memory_output_dir)


## 5. Download this mode's results

Run the next cell. Download the ZIP from Kaggle's Output/files panel. Keep both the FP16 and NF4 ZIPs and send them back together.

Before running the other mode, download this ZIP and restart the Python kernel. If your Kaggle action discards the whole session rather than restarting just the kernel, download first so the files are preserved.


In [ ]:
import shutil
from pathlib import Path

zip_path = Path(shutil.make_archive(
    str(memory_output_dir), "zip", root_dir=memory_output_dir
))
print("Download:", zip_path)
print("Bytes:", zip_path.stat().st_size)


## What to send back

Send the two generated ZIPs. Each contains:

| File | Meaning |
| --- | --- |
| `metadata.json` | Mode, pinned checkpoint, software/GPU, resident baseline, measurement notes and completion checks |
| `memory_trials.csv` | Each request's baseline, operation peaks, KV growth, cleanup result and conservative overhead bounds |
| `memory_snapshots.csv` | Ordered observations of allocated/reserved/device memory at each lifecycle stage |
| `cache_tensors.csv` | Layer-by-layer actual cache shapes, dtype, logical size and backing storage |
| `memory_summary.csv` | Per-shape summary of successful measurements and cleanup checks |

Expected healthy result: `run_status = "completed"`, 48 successful trials, all cleanup checks true, and 28,672 bytes of KV growth per request for one decode step. For batch 8, that growth is 229,376 bytes in total. A cleanup difference within 1 MiB is flagged as passing; the exact difference is still saved. Investigate nonzero differences before assuming they are harmless.

## How to interpret the numbers

- **Allocated** is live tensor allocation; **reserved** includes PyTorch's allocator pool. They overlap and must not be added. `empty_cache()` trims unused pool memory; it does not release live cache tensors.
- **KV logical bytes** describe the active tensor contents. **Unique backing bytes** count underlying storage once, including when tensors are views. Neither automatically equals physical allocator-page consumption.
- **Resident baseline** is model-plus-persistent tensor allocation with no active request. It includes quantization tensors and persistent buffers; it is not an isolated weight-only byte count.
- **Whole-device free/used memory** also reflects the CUDA context and allocations outside this process. It can fluctuate. It does not alone establish bytes reclaimable for restored weights.
- **Operation peaks already contain the model, cache, inputs and temporaries.** Do not add those components again. Raw snapshots preserve the aligned observations needed to reason about them.
- `*_non_kv_upper_bound_bytes` is an intentionally conservative bound, not exact workspace: it subtracts only the minimum cache guaranteed to exist throughout that operation. For prefill that minimum is zero, so the bound includes cache construction. Use observed total peaks to check supported shapes; refine the bound before claiming a tight serving capacity.
- This tests the Transformers **DynamicCache** used by your fixed-mode profilers. It does not demonstrate a resizable paged KV arena, a precision switch, or memory-safe weight restoration. Switching time, staging memory, host weight copies, allocator rules and the final capacity budget remain the next step.

The notebook was checked locally with simulated tensor/cache and CUDA-counter fixtures, including cleanup and error handling. Actual model execution and GPU memory values must be checked on your Kaggle T4; no CUDA device was available for the local checks.

Documentation: [PyTorch memory management](https://docs.pytorch.org/docs/2.11/notes/cuda.html#memory-management), [peak-counter reset](https://docs.pytorch.org/docs/2.11/generated/torch.cuda.memory.reset_peak_memory_stats.html), [Transformers caching](https://huggingface.co/docs/transformers/v5.20.2/cache_explanation).
